### Imports

In [ ]:
from pathlib import Path

import spikeinterface.full as si
import spikeinterface.widgets as sw
from probeinterface import get_probe
from probeinterface.plotting import plot_probe
import matplotlib.pyplot as plt

%matplotlib widget

### Data Loading and Exploration

In [ ]:
file_path="../data/raw/sub-KM131_ses-20180116T184757_behavior+ecephys+image.nwb"

In [ ]:
# Load the ElectricalSeries contained in the NWB file as a
# SpikeInterface Recording object.
#
# This operation creates a lazy extractor: the complete voltage signal
# is not immediately loaded into memory.
recording = si.read_nwb_recording(
    file_path=file_path,
    load_channel_properties=True,
)

recording

### The SpikeInterface `Recording` object

A SpikeInterface Recording object provides a standardized representation of a continuous, multichannel electrophysiology recording. It acts as an interface between the original data format—such as NWB, SpikeGLX, Open Ephys, or a binary file—and the subsequent analysis pipeline. https://spikeinterface.readthedocs.io/en/stable/tutorials/core/plot_1_recording_extractor.html?utm_source=chatgpt.com


recording = si.read_nwb_recording("recording.nwb")  

recording = si.read_spikeglx("spikeglx_folder/") 

 recording = si.read_openephys("openephys_folder/") 

The object does not normally load the complete signal into memory. Instead, it stores the information needed to access specific channels and time intervals when requested. This is known as lazy loading.

A Recording object contains:

- the number and identifiers of the channels;
- the sampling frequency;
- the recording duration and number of segments;
- the data type and signal scaling;
- channel-level properties imported from the source data;
- optionally, the probe geometry and channel-to-contact mapping.

The continuous voltage data can be accessed with recording.get_traces(), while metadata can be inspected using methods such as get_num_channels(), get_sampling_frequency(), get_channel_ids(), and get_property_keys().

### The SpikeInterface Probe

[ProbeInterface](https://probeinterface.readthedocs.io/) is a Python package for describing and handling neural probes. It stores information such as:

* contact positions and geometry;
* contact shapes and dimensions;
* probe manufacturer and model;
* channel-to-contact mapping;
* probe wiring.

#### Load a registered probe

ProbeInterface provides a public library containing probe geometries from several manufacturers:

```python
from probeinterface import get_probe

probe = get_probe(
    manufacturer="cambridgeneurotech",
    probe_name="ASSY-77-H3",
)

probe
```

The probe can be visualized with:

```python
from probeinterface.plotting import plot_probe

plot_probe(
    probe,
    with_contact_id=True,
    with_device_index=True,
)
```

#### Generate a simple probe

If the exact probe model is unavailable, a simple linear probe can be generated using the number of recording channels:

```python
from probeinterface.generator import generate_linear_probe

probe = generate_linear_probe(
    num_elec=recording.get_num_channels(),
    ypitch=20,
    contact_shapes="rect",
    contact_shape_params={
        "width": 11,
        "height": 15,
    },
)
```

ProbeInterface also provides a predefined dummy probe for testing:

```python
from probeinterface.generator import generate_dummy_probe

dummy_probe = generate_dummy_probe()
```

Before attaching a probe to a recording, each physical contact must be associated with the correct acquisition channel:

```python
import numpy as np

probe.set_device_channel_indices(
    np.arange(recording.get_num_channels())
)

recording = recording.set_probe(
    probe,
    in_place=False,
)
```

> Using `np.arange(...)` assumes that recording channels and physical probe contacts have the same order. For real experimental data, the channel mapping should be verified before attaching the probe.


In [ ]:
from probeinterface import get_probe

probe = get_probe(
    manufacturer="cambridgeneurotech",
    probe_name="ASSY-77-H3",
)
probe.wiring_to_device('ASSY-77>Adpt.A64-Om32_2x-sm-NN>RHD2164')
probe

In [ ]:
recording

In [ ]:
recording= recording.set_probe(probe)
recording

In [ ]:
# Plot the physical geometry of the probe.
#
# `with_contact_id=True` displays the physical contact identifiers.
# `with_device_index=True` displays the corresponding acquisition-channel
# indices, if a wiring configuration is available.

from probeinterface.plotting import plot_probe

fig, ax = plt.subplots()

plot_probe(
    probe,
    ax=ax,
    #with_contact_id=True,
    #with_device_index=True,
    ylims=(0,1000)
)

ax.set_ylim(0, 1000)

In [ ]:
print(f"Number of contacts: {probe.get_contact_count()}")
print(f"Dimensions: {probe.ndim}D")
print(f"Units: {probe.si_units}")
print(f"Contact positions shape: {probe.contact_positions.shape}")
print(f"Device mapping: {probe.device_channel_indices}")

### Visualizing the data

In [ ]:
start_time = recording.get_times()[0]

w = sw.plot_traces(
    recording,
    time_range=(start_time, start_time + 10),
    return_in_uV=True,
    order_channel_by_depth=True
)
plt.show()

w.ax.set_title("test")


### Preprocessing

In [ ]:
#phase_shift_recording = si.phase_shift(recording)

filtered_recording = si.bandpass_filter(
    recording, freq_min=300, freq_max=6000
)

cmr_recording = si.common_reference(
    filtered_recording, operator="median"
)

#print(phase_shift_recording)
print(filtered_recording)
print(cmr_recording)

In [ ]:
si.plot_traces(
    cmr_recording,
    time_range=(start_time, start_time + 0.1),
    return_in_uV=True,
    order_channel_by_depth=True
)
plt.show()

In [ ]:
output_folder = Path("../reports/processed/dendi")

first_shank_recording = cmr_recording

sorting = si.run_sorter(
    "kilosort4",
    first_shank_recording,
    folder=output_folder,
    remove_existing_folder=True,
    do_CAR=False,
    highpass_cutoff=0.1,
    verbose=True,
)